# bce-log-loss-real-fake — worked example 1: Discriminator loss written out as the raw log-form

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `bce-log-loss-real-fake`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The discriminator's objective is to maximize `log(D(real)) + log(1 - D(fake))`, so the loss minimizes the negative of that. When the targets are constant 1s and 0s you don't need `nn.BCELoss` at all — you can write the two `-log(...)` terms directly. This worked example shows the manual log-form so you can see what `F.binary_cross_entropy` is really computing under the hood.

## Worked solution

**Goal:** compute `-log(D(real)).mean() - log(1 - D(fake)).mean()` by hand.

1. **Real term.** D wants to output a probability near 1 on real images. The per-sample penalty is `-log(d_pred_real)`: zero when the prediction is 1, large when it heads toward 0. We take `.mean()` over the batch.
2. **Fake term.** D wants to output near 0 on fakes. The matching penalty is `-log(1 - d_pred_fake)`: zero when the prediction is 0, large as it climbs toward 1.
3. **Why a small epsilon.** `log(0)` is `-inf`. Real sigmoid outputs are never exactly 0 or 1, but clamping into `[eps, 1-eps]` keeps the demo numerically safe and matches what stable BCE kernels do internally.
4. **Sum the means.** The combined discriminator loss is the sum of the two mean terms — a single scalar.
5. **Cross-check.** Computing the same thing with `F.binary_cross_entropy` against `ones_like` / `zeros_like` targets must agree, because BCE with a target of 1 is exactly `-log(p)` and with a target of 0 is exactly `-log(1-p)`.

In [ ]:
import torch.nn.functional as F

def manual_disc_loss(d_pred_real: t.Tensor, d_pred_fake: t.Tensor) -> t.Tensor:
    eps = 1e-7
    pr = d_pred_real.clamp(eps, 1 - eps)
    pf = d_pred_fake.clamp(eps, 1 - eps)
    real_term = -t.log(pr).mean()
    fake_term = -t.log(1 - pf).mean()
    return real_term + fake_term

t.manual_seed(0)
d_pred_real = t.rand(8)
d_pred_fake = t.rand(8)

manual = manual_disc_loss(d_pred_real, d_pred_fake)
bce = (F.binary_cross_entropy(d_pred_real, t.ones_like(d_pred_real))
       + F.binary_cross_entropy(d_pred_fake, t.zeros_like(d_pred_fake)))
print('manual log-form loss:', round(manual.item(), 6))
print('F.binary_cross_entropy:', round(bce.item(), 6))
print('match:', t.allclose(manual, bce, atol=1e-5))